# 1. Processing the data

This notebook turns the raw model files and ERA5 into the seasonal means that the other notebooks start from. It runs on JASMIN, once per variable.

| Step | From | To (under `SCRATCH`, then `DATA_DIR`) |
|---|---|---|
| 1. LESFMIP to monthly stores | one netCDF file per member | `monthly/lesfmip/<variable>/<model>_<variable>_<experiment>_interp_monthly.zarr` |
| 2. ERA5 to monthly means | hourly ERA5 analyses | `monthly/era5_<variable>.zarr`, on the LESFMIP grid |
| 3. Seasonal means | the monthly stores | `seasonal/lesfmip_<variable>.zarr` and `seasonal/era5_<variable>.zarr` |
| 4. Test sample | the monthly stores | `tests/data/*_sample.nc` in this repository |
| 5. Sea ice (once) | the sea-ice concentration files | `results/siconc/full/extent.zarr` and `climatology.zarr` |

**Where things are written.** Dask writes zarr stores from many workers at once, which the group workspaces do not support. So everything is written under `paths.SCRATCH` and moved to `paths.DATA_DIR` by hand (the last section has the commands). Everything that reads data looks in `DATA_DIR` first and then in `SCRATCH`, so a store can be used as soon as it is written.

Steps 1 and 2 take hours and only need running again for a new variable or new models; both skip what already exists. Sections marked (JASMIN) need the data there. Section 3 can also run on the small sample in `tests/data`, which is a quick way to see what each step does.

Each step prints what it made, with a **quick look** (a plain xarray plot, all its code here) before each method figure. Names end in their type: `_tree` (DataTree), `_ds` (Dataset), `_da` (DataArray), `_arr` (numpy array), `_df` (DataFrame).

## Setup

In [ ]:
import sys
sys.path[:0] = ['..', '../../extant-functions']   # this repository (the extant package) and plotting_modules

import logging
import warnings
from importlib import reload

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from dask.distributed import wait
from plotting_modules import formatting, maps
from extant import config, loading, paths, preprocessing as pp, storage
from extant.logger import setup_logging
from extant.plots import methods

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

#(t): Dask warns about every large task graph; the graphs here are large on purpose
warnings.filterwarnings('ignore', message='.*Sending large graph of size.*', category=UserWarning)

from importlib.util import find_spec

from extant import convert, datatree, sea_ice
from extant.plots import sea_ice as sip

#(t): flox speeds up groupby/resample on dask (the seasonal means); xarray uses it automatically when installed
if find_spec('flox') is None:
    logger.warning('flox is not installed: pip install flox to speed up the seasonal means')

## Settings

`VARIABLE` is the CMIP name of the variable to process. Its units, and how to convert them, are in `config.VARIABLES`: temperature goes from K to °C, $T_{°C} = T_K - 273.15$, and precipitation from kg m⁻² s⁻¹ to mm day⁻¹, $P_{mm\,day^{-1}} = 86400\,P_{kg\,m^{-2}\,s^{-1}}$.

In [ ]:
VARIABLE = 'tas'
var = config.VARIABLES[VARIABLE]
var

In [ ]:
#(t): Where this session reads from and writes to
print(f'reading raw LESFMIP from {paths.lesfmip_raw(VARIABLE)}')
print(f'writing to               {paths.SCRATCH}')
print(f'reading outputs from     {paths.DATA_DIR}, then {paths.SCRATCH}')

## Dask cluster (JASMIN)

The heavy steps run on a Dask Gateway cluster on JASMIN. `jasmin.stop_all(gateway)` stops every cluster left over from an earlier kernel.

In [ ]:
#(t): Start a Dask Gateway cluster, or reuse the one already running, and send it the extant package
#(c): Re-run upload_package after editing anything in extant/: the workers run its functions
from extant import jasmin

gateway, cluster, client = jasmin.connect(n_workers=20, wait_for=6)
jasmin.upload_package(client)
client

## 1. LESFMIP: member files to monthly stores (JASMIN)

The raw files are laid out as `<variable>/<experiment>/<model>/*.nc`, one file per member. Each model's members for one experiment are opened together and stacked along a new `member` dimension, labelled from the file names (`r1i1p1f1` and the like; `convert.member_from_path`). Only the area south of 40°S is kept.

First, which models have which experiments. Only models with at least `config.MIN_EXPERIMENTS` of the experiments are converted.

In [ ]:
#(t): Models with raw files for each experiment, the best-covered first
availability_df = convert.availability(VARIABLE, experiments=config.MAIN_EXPERIMENTS)
availability_df.assign(total=availability_df.sum(axis=1))

In [ ]:
convert_models = list(availability_df.index[availability_df.sum(axis=1) >= config.MIN_EXPERIMENTS])
convert_models

Now convert them. A store that already exists (in either place) is skipped, so this can be re-run after an interruption; `overwrite=True` redoes everything. A model whose files will not open together is reported in `failures` rather than stopping the loop.

In [ ]:
%%time
#(t): Every model and experiment to monthly stores (hours, the first time)
failures = convert.convert_lesfmip(VARIABLE, convert_models, config.MAIN_EXPERIMENTS, overwrite=False)
{key: {group: str(failure['error'])[:200] for group, failure in groups.items()} for key, groups in failures.items()}

A failed group can be retried after removing its broken files. For example, member 55 of historical GISS-E2-1-G has no time axis:

```python
failed = failures[('historical', 'GISS-E2-1-G')]['interp']
broken = failed['files'].pop(55)
convert.save_members(failed['files'], failed['store'], VARIABLE)
```

## 2. ERA5: hourly files to monthly means (JASMIN)

Each month of hourly analyses is averaged on a worker (summing in float64), then regridded **conservatively** onto the LESFMIP 2.5° grid with xesmf, so each model grid cell gets the area-weighted mean of the ERA5 cells inside it. Each year is saved as its own store as soon as its twelve months are done, so an interrupted run picks up where it stopped. At the end the year stores are combined into one store and checked (every month once, in order).

In order: the grid to regrid onto, which years are left, one month taken through step by step with checks of the regridding, then every year left.

ERA5 is kept on its own (standard) calendar. Its seasons are matched to the models' by their (year, season) labels in step 3, not by date.

### The grid to regrid onto

Every model's monthly store should be on the same 2.5° grid (the interpolated files), south of 40°S. It is taken from one of them, and the rest are checked against it.

In [ ]:
#(t): The LESFMIP grid, from one monthly store, and a check that every other store is on it
monthly_tree = loading.open_lesfmip_monthly(VARIABLE, groups=[config.GROUP])
target_ds = monthly_tree.leaves[0].to_dataset()[['lat', 'lon']]
on_grid = {node.path: np.array_equal(node['lat'], target_ds.lat) and np.array_equal(node['lon'], target_ds.lon)
           for node in monthly_tree.leaves}
print(f'grid of {monthly_tree.leaves[0].path}; {sum(on_grid.values())} of {len(on_grid)} stores on it, '
      f'not on it: {[path for path, same in on_grid.items() if not same] or "none"}\n')
for dim in ('lat', 'lon'):
    print(f'{dim}: {target_ds.sizes[dim]} cells, {float(target_ds[dim][0])} to {float(target_ds[dim][-1])}, '
          f'every {float(target_ds[dim].diff(dim).mean())}°')
target_ds

### What is left to do

A year already saved, either as a year store (an interrupted run) or in the combined store, is skipped.

In [ ]:
#(t): The years wanted, those already done and those left
ERA5_YEARS = range(config.ERA5_START_YEAR, 2025)
years_dir = paths.era5_years_dir(VARIABLE)
stored_years = sorted(int(p.name.removesuffix('.zarr')) for p in paths.find_all(years_dir, '*.zarr'))
combined_years = (sorted(set(storage.open_dataarray(paths.era5_monthly(VARIABLE)).time.dt.year.values.tolist()))
                  if storage.exists(paths.era5_monthly(VARIABLE)) else [])
todo_years = [year for year in ERA5_YEARS if year not in stored_years and year not in combined_years]
print(f'wanted:                {ERA5_YEARS[0]}-{ERA5_YEARS[-1]} ({len(ERA5_YEARS)} years)')
print(f'in the combined store: {combined_years}')
print(f'saved as year stores:  {stored_years}')
print(f'left to do:            {todo_years}')

In [ ]:
#(t): The hourly files of each month left (years down, months across). Every month should have 24 a day
month_files = {year: {month: convert.era5_files(VARIABLE, year, month) for month in range(1, 13)} for year in todo_years}
hours_df = pd.DataFrame({year: {month: len(files) for month, files in months.items()}
                         for year, months in month_files.items()}).T
expected_df = pd.DataFrame({year: {month: 24 * pd.Timestamp(year, month, 1).days_in_month for month in range(1, 13)}
                            for year in todo_years}).T
print(f'{int(hours_df.sum().sum())} hourly files under {paths.ERA5_RAW}; '
      f'years with missing hours: {list(hours_df.index[hours_df.ne(expected_df).any(axis=1)]) or "none"}')
hours_df

### One month, step by step

January of the first year left (of the first year wanted, if none is left), done here on its own with checks. The loop below does every month the same way.

ERA5 is global at 0.25°. Each hourly file is sorted south to north and cut at `convert.ERA5_LAT_MAX`, a little north of the LESFMIP edge, so the edge cells are fully covered.

In [ ]:
#(t): The example month's files, and one hour of it as the worker reads it
example_year = (todo_years or list(ERA5_YEARS))[0]
example_files = convert.era5_files(VARIABLE, example_year, 1)
print(f'{len(example_files)} files for {example_year}-01:\n  {example_files[0]}\n  ...\n  {example_files[-1]}\n')
hour_ds = convert.era5_south(xr.open_dataset(example_files[0]))
hour_ds

The regridder needs the edges of every cell on both grids. They are put half-way between the cell centres and clipped at ±90°: the LESFMIP row at 90°S would otherwise run to 91.25°S, past the pole.

In [ ]:
#(t): Both grids with their cell edges (lat_bounds, lon_bounds); the first and last edges are in the printout
import cf_xarray  # noqa: F401  (adds .cf)
import xesmf

era5_grid_ds = hour_ds[['latitude', 'longitude']].rename(latitude='lat', longitude='lon').cf.add_bounds(['lat', 'lon'])
target_grid_ds = target_ds.cf.add_bounds(['lat', 'lon'])
for grid_ds in (era5_grid_ds, target_grid_ds):
    grid_ds['lat_bounds'] = grid_ds['lat_bounds'].clip(-90.0, 90.0)
print(era5_grid_ds, '\n')
print(target_grid_ds)

In [ ]:
%%time
#(t): The conservative regridder: each ERA5 cell's weight in a LESFMIP cell is the area they share
regridder = xesmf.Regridder(era5_grid_ds, target_grid_ds, 'conservative', periodic=True)
regridder

In [ ]:
%%time
#(t): The example month, averaged on a worker, on the ERA5 grid and then regridded
example_month_da = xr.DataArray(client.submit(convert.era5_month_mean, example_files, var.era5_name).result(),
                                dims=('lat', 'lon'), coords={'lat': era5_grid_ds.lat.values, 'lon': era5_grid_ds.lon.values},
                                name=VARIABLE)
example_regridded_da = regridder(example_month_da)
print(example_month_da, '\n')
print(example_regridded_da)

**Checks.**

1. The regridded month is on exactly the LESFMIP grid.
2. A field of ones regrids to one in every cell, i.e. every LESFMIP cell is fully covered by ERA5. Less than one in a row means the cell edges are wrong there (at the pole, or at the northern edge if ERA5 were cut too far south).
3. Conservative regridding keeps the area-weighted mean: it is the same over the LESFMIP domain before and after.
4. By hand: the cell at `config.POINT` is the mean of the ERA5 cells, each weighted by the area it shares with the cell.

On a lat-lon grid the area of a box is $R^2\,\Delta\lambda\,\Delta(\sin\varphi)$, which is all `area_inside` works out. ESMF takes cell edges as great circles rather than lines of latitude, so its areas differ very slightly from these, and 3 and 4 agree closely rather than exactly.

In [ ]:
def area_inside(grid_ds, lat_range=(-90, 90), lon_range=(-360, 720)):
    #(t): The area of each cell of grid_ds inside the box lat_range x lon_range (degrees), in units of R²
    south, north = np.sin(np.deg2rad(grid_ds.lat_bounds.clip(*lat_range))).values.T
    west, east = np.deg2rad(grid_ds.lon_bounds.clip(*lon_range)).values.T
    return (xr.DataArray(north - south, dims='lat', coords={'lat': grid_ds.lat.values})
            * xr.DataArray(east - west, dims='lon', coords={'lon': grid_ds.lon.values}))


#(t): 1. On the LESFMIP grid
assert np.array_equal(example_regridded_da.lat, target_ds.lat) and np.array_equal(example_regridded_da.lon, target_ds.lon)

#(t): 2. Ones stay ones
ones_da = regridder(xr.ones_like(example_month_da))
print(f'ones regridded: {float(ones_da.min()):.6f} to {float(ones_da.max()):.6f}; rows not 1: '
      f'{ones_da.lat.values[(abs(ones_da - 1) > 1e-6).any("lon").values].tolist() or "none"}')

#(t): 3. The area-weighted mean over the LESFMIP domain (for ERA5, the part of each cell inside it)
target_area_da = area_inside(target_grid_ds)
domain = (float(target_grid_ds.lat_bounds.min()), float(target_grid_ds.lat_bounds.max()))
before = float(example_month_da.weighted(area_inside(era5_grid_ds, domain)).mean())
after = float(example_regridded_da.weighted(target_area_da).mean())
print(f'area-weighted mean, {domain[0]} to {domain[1]}°: ERA5 {before:.4f}, regridded {after:.4f}, '
      f'difference {after - before:.1e}')

#(t): 4. By hand at the example point
cell_ds = target_grid_ds.sel(**config.POINT)
by_hand = float(example_month_da.weighted(area_inside(era5_grid_ds, cell_ds.lat_bounds.values,
                                                      cell_ds.lon_bounds.values)).mean())
regridded = float(example_regridded_da.sel(**config.POINT))
print(f'cell at {float(cell_ds.lat)}°, {float(cell_ds.lon)}° (edges {cell_ds.lat_bounds.values}, '
      f'{cell_ds.lon_bounds.values}): regridded {regridded:.4f}, by hand {by_hand:.4f}, '
      f'difference {regridded - by_hand:.1e}')

In [ ]:
#(t): Quick look: the month on the ERA5 grid and on the LESFMIP grid, on one colour scale
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True, layout='constrained')
scale = dict(vmin=float(example_month_da.min()), vmax=float(example_month_da.max()), cmap='RdYlBu_r')
example_month_da.plot(ax=axes[0], **scale)
example_regridded_da.plot(ax=axes[1], **scale)
axes[0].set_title(f'ERA5 0.25°, {example_year}-01')
axes[1].set_title('regridded to the LESFMIP 2.5° grid');

In [ ]:
#(t): Quick look: the zonal means, before and after
example_month_da.mean('lon').plot(label='ERA5 0.25°')
example_regridded_da.mean('lon').plot(marker='o', ls='', label='regridded 2.5°')
plt.legend();

### All of it

Every month left goes to the workers at once. As each year's twelve months come back they are put on the ERA5 grid, regridded with the regridder above and saved as a year store, and the year's range and area-weighted mean (raw units, K for temperature) are printed: a year far off the others has gone wrong.

In [ ]:
%%time
#(t): Every month left, averaged on the workers; each year regridded and saved as soon as its 12 months are back
futures = {year: [client.submit(convert.era5_month_mean, month_files[year][month], var.era5_name, pure=False)
                  for month in range(1, 13)]
           for year in todo_years}
for year, year_futures in futures.items():
    year_da = xr.DataArray(np.stack(client.gather(year_futures)), dims=('time', 'lat', 'lon'),
                           coords={'time': pd.date_range(f'{year}-01-01', periods=12, freq='MS'),
                                   'lat': era5_grid_ds.lat.values, 'lon': era5_grid_ds.lon.values})
    year_regridded_da = regridder(year_da).astype('float32').rename(VARIABLE)
    storage.save(year_regridded_da, years_dir / f'{year}.zarr')
    print(f'{year}: {int(hours_df.loc[year].sum())} hours, {float(year_regridded_da.min()):.1f} to '
          f'{float(year_regridded_da.max()):.1f}, mean {float(year_regridded_da.weighted(target_area_da).mean()):.2f}, '
          f'{int(year_regridded_da.isnull().sum())} missing')

In [ ]:
#(t): Join the year stores (and the combined store, if there is one) into one store, check it (every month once,
#(t): in order, the saved copy equal to the data) and delete the year stores
era5_monthly_da = convert.combine_era5_years(VARIABLE)
era5_monthly_da

In [ ]:
#(t): Check: every year wanted is there, each with 12 months
months_per_year = era5_monthly_da.time.dt.year.to_series().value_counts().sort_index()
print(f'{len(months_per_year)} years, {months_per_year.index[0]} to {months_per_year.index[-1]}')
print(f'years wanted but missing: {sorted(set(ERA5_YEARS) - set(months_per_year.index)) or "none"}')
print(f'years without 12 months:  {list(months_per_year.index[months_per_year != 12]) or "none"}')

In [ ]:
#(t): Quick look: the monthly means at the example point
era5_monthly_da.sel(**config.POINT).plot(figsize=(14, 3));

## 3. Monthly to seasonal means

### Open the monthly data

Run **one** of the next two sections. Both leave the same variables: `lesfmip_tree`, monthly LESFMIP as a DataTree laid out `/<model>/<experiment>`, and `era5_monthly_da`, both in the units of the raw files.

#### Sample data

The 3×3 block of grid cells around `config.POINT` in `tests/data`: two models, every experiment.

In [ ]:
lesfmip_tree = loading.open_lesfmip_sample(experiments=config.MAIN_EXPERIMENTS)
era5_monthly_da = loading.open_era5_sample()
lesfmip_tree

#### Full data (JASMIN)

The `config.N_MODELS` models with stores for the most experiments.

In [ ]:
#(t): Which models have a monthly store for which experiment
stores_df = loading.store_table(VARIABLE, groups=[config.GROUP])
stores_df

In [ ]:
models = loading.models_with_most_experiments(stores_df, config.N_MODELS, experiments=config.MAIN_EXPERIMENTS)
lesfmip_tree = loading.open_lesfmip_monthly(VARIABLE, models=models, experiments=config.MAIN_EXPERIMENTS,
                                            groups=[config.GROUP])
era5_monthly_da = loading.open_era5_monthly(VARIABLE)
lesfmip_tree

### What is there

How many members each model ran for each experiment. A model's members differ only in their initial conditions, so the spread across them is the model's internal variability, and the more members, the better the forced response and the variability are pinned down.

In [ ]:
#(t): Members per model (rows) and experiment (columns)
member_counts_df = pd.Series({(node.parent.name, node.name): node.sizes['member'] for node in lesfmip_tree.leaves})
member_counts_df = member_counts_df.unstack()[[e for e in config.MAIN_EXPERIMENTS if e in member_counts_df.index.levels[1]]]
member_counts_df

In [ ]:
fig, ax = methods.member_count_heatmap(member_counts_df, title='LESFMIP members per model and experiment')

In [ ]:
#(t): Every node's dimensions, size in memory and chunking
pd.DataFrame.from_dict({node.path: datatree.coord_info(node.to_dataset()) for node in lesfmip_tree.leaves},
                       orient='index')

In [ ]:
#(t): Two months of one member, to see the domain (everything south of 40°S)
months_da = pp.to_analysis_units(lesfmip_tree[config.PLOT_SEL['model']][config.PLOT_SEL['experiment']].to_dataset()
                                 .isel(member=0, time=[0, 6]), VARIABLE)[VARIABLE].load()
print(months_da)

In [ ]:
#(t): Quick look: the two months on the plain lat-lon grid
months_da.plot(x='lon', y='lat', col='time');

In [ ]:
levels = np.linspace(float(months_da.quantile(0.02)), float(months_da.quantile(0.98)), 15)
panels = maps.polar_grid(months_da, col_dim='time', levels=levels, cmap='RdYlBu_r',
                         label_fmt={'time': lambda t: str(t)[:7]}, cbar_label=var.label, tag=False)

### How a seasonal mean is made

Each season is the mean of its three monthly means (each month counts equally, whatever its length):

$$
\bar{x}_{\mathrm{DJF},\,y} = \tfrac{1}{3}\left(x_{\mathrm{Dec}\,y} + x_{\mathrm{Jan}\,y+1} + x_{\mathrm{Feb}\,y+1}\right),
\qquad
\bar{x}_{\mathrm{MAM},\,y} = \tfrac{1}{3}\left(x_{\mathrm{Mar}\,y} + x_{\mathrm{Apr}\,y} + x_{\mathrm{May}\,y}\right), \ \ldots
$$

so DJF is labelled by the year of its December, and a "year" runs from March to the next February. Seasons with fewer than three months are dropped (the January–February stub at the start of a record), and so is any year without all four seasons. Every experiment is first cut at the end of 2014, where historical ends, so the record ends with DJF 2013 and "the final 21 years" are the same years in every experiment. The time axis is then split into `year` × `season` (`pp.split_time`), so that a rolling window moves through the years of one season.

The figure: one member at `config.POINT`, monthly means (grey) and the seasonal means made from them.

In [ ]:
#(t): One member at the example point: monthly (in analysis units) on time, and its seasonal means on (year, season)
point_monthly_da = pp.to_analysis_units(lesfmip_tree[config.PLOT_SEL['model']][config.PLOT_SEL['experiment']]
                                        .to_dataset().sel(**config.POINT), VARIABLE)[VARIABLE].isel(member=0).load()
point_seasonal_da = pp.seasonal_means(point_monthly_da.sel(time=slice(None, config.LAST_MONTH)))
print(point_monthly_da, '\n')
print(point_seasonal_da)

In [ ]:
#(t): Check by hand: DJF 1990 is the mean of December 1990, January 1991 and February 1991
months_arr = point_monthly_da.sel(time=slice('1990-12', '1991-02')).values
print('Dec, Jan, Feb:', months_arr.round(3), ' mean:', months_arr.mean().round(3))
print('DJF 1990:     ', point_seasonal_da.sel(year=1990, season='DJF').values.round(3))

In [ ]:
#(t): Quick look: the seasonal means, one line per season
point_seasonal_da.plot.line(x='year', hue='season');

In [ ]:
fig, ax = methods.seasonal_mean_demo(point_monthly_da, point_seasonal_da, slice(1990, 1995), units=var.units)

In [ ]:
#(t): The same seasonal means, before and after splitting time into year x season
fig, axes = methods.year_season_demo(point_seasonal_da, units=var.units)

### All of it

`pp.seasonal_tree` runs these steps on every model and experiment, and `pp.seasonal_era5` on ERA5, which it also puts on the LESFMIP grid (`pp.match_grid`) and starts in 1979 (`config.ERA5_START_YEAR`), the start of the satellite era. Both are lazy: nothing is computed until the result is saved or loaded.

In [ ]:
lesfmip_season_tree = pp.seasonal_tree(lesfmip_tree, VARIABLE)
era5_seasonal_da = pp.seasonal_era5(era5_monthly_da, VARIABLE, like=lesfmip_season_tree)
lesfmip_season_tree

In [ ]:
#(t): ERA5's seasonal means, on (year, season, lat, lon)
era5_seasonal_da

### Save (JASMIN)

Writing computes everything, in parallel on the cluster. The other notebooks then open these two stores in seconds.

In [ ]:
%%time
storage.save(lesfmip_season_tree, paths.seasonal('lesfmip', VARIABLE))
storage.save(era5_seasonal_da, paths.seasonal('era5', VARIABLE))

In [ ]:
#(t): Reopen what was saved, for the checks below
seasonal_path = paths.find(paths.seasonal('lesfmip', VARIABLE))
print(f'reading {seasonal_path}')
lesfmip_season_tree = xr.open_datatree(seasonal_path, engine='zarr', chunks={})
era5_seasonal_da = storage.open_dataarray(paths.seasonal('era5', VARIABLE))

### Checks

The seasonal means at the example point: the members of one model (grey) and ERA5 (black) over the years they share. ERA5 should sit among the members; a constant offset would be a mean-state bias of the model, which the ERA5 notebook tests properly.

In [ ]:
#(t): ERA5 among one model's historical members at the example point, every season
point_members_da = lesfmip_season_tree[config.PLOT_SEL['model']][config.PLOT_SEL['experiment']][VARIABLE].sel(**config.POINT)
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharex=True, layout='constrained')
for ax, season in zip(axes, ['DJF', 'MAM', 'JJA', 'SON']):
    members_da = point_members_da.sel(season=season, year=slice(config.ERA5_START_YEAR, None))
    ax.plot(members_da.year, members_da.transpose('year', 'member'), color='0.6', lw=0.4, alpha=0.5)
    ax.plot(era5_seasonal_da.year, era5_seasonal_da.sel(**config.POINT).sel(season=season), color='k', lw=1.5)
    ax.set_title(season)
    ax.set_xlabel('Year')
axes[0].set_ylabel(var.label)
fig.suptitle(f"{config.PLOT_SEL['model']} {config.PLOT_SEL['experiment']} members (grey) and ERA5 (black) "
             f"at {formatting.format_latlon(config.POINT)}");

## 4. Test sample (JASMIN)

The 3×3 block of grid cells around `config.POINT`, from two models with every experiment and from ERA5, saved to `tests/data`. The tests and the sample sections of every notebook use it. It is temperature only, so only re-run this for `tas`.

In [ ]:
sample_tree = loading.open_lesfmip_monthly(VARIABLE, models=['CanESM5', 'HadGEM3-GC31-LL'], groups=[config.GROUP])
written = loading.write_sample(sample_tree, loading.open_era5_monthly(VARIABLE))
[f'{f.name}: {f.stat().st_size / 1e6:.1f} MB' for f in written]

## 5. Sea ice (JASMIN)

The figures mark the sea-ice edge, as Bracegirdle et al. (2024, *npj Clim. Atmos. Sci.* 7, 276) do: on maps as the 15% contour of the ensemble-mean concentration, and on the zonal-mean figures as an **equivalent latitude**, the latitude the edge would be at if the ice and the continent it surrounds were a cap round the pole. The area of the Earth south of latitude $\varphi$ is

$$
A(\varphi) = 2\pi R^2\,(1 + \sin\varphi),
$$

so the edge is at the latitude whose cap holds the sea-ice extent and the continent together:

$$
\varphi_{\mathrm{eq}} = \arcsin\left(\frac{\mathrm{SIE} + A_{\mathrm{continent}}}{2\pi R^2} - 1\right).
$$

- **Sea-ice extent** (SIE): the area of the cells with at least 15% ice, worked out every month and averaged into seasons, for every member.
- **The continent**: the land south of 60°S, i.e. the cells where the ocean model gives no concentration. With no sea ice at all, $\varphi_{\mathrm{eq}}$ is about 71°S.

Each model keeps its own edge; nothing is averaged across models. Unlike the paper, which uses each model's own ocean grid, everything here is on the common 2.5° grid of the interpolated files.

This section runs once, whatever `VARIABLE` is. It converts the concentration (`siconc`) of the models analysed, then saves each member's seasonal extent and the ensemble-mean concentration over the final `config.WINDOW` years, as `results/siconc/full/extent.zarr` and `climatology.zarr`. Notebook 04 reads them. The code is in `extant/sea_ice.py`.

**Where the files are.** `paths.lesfmip_raw('siconc')`, built from the variable's CMIP table in `config.VARIABLES`. The table (`SImon`) is a guess: change it there if the interpolated sea-ice files are somewhere else.

In [ ]:
#(t): The models analysed (those in the seasonal store of VARIABLE), and which of them have sea-ice files
SEA_ICE = 'siconc'
ice_models = list(loading.open_seasonal(VARIABLE).children)
print(f'reading raw sea ice from {paths.lesfmip_raw(SEA_ICE)}')
ice_availability_df = convert.availability(SEA_ICE, experiments=config.MAIN_EXPERIMENTS)
ice_availability_df.reindex(ice_models)

In [ ]:
%%time
#(t): Concentration files to monthly stores, as in section 1 (skips what exists)
ice_failures = convert.convert_lesfmip(SEA_ICE, ice_models, config.MAIN_EXPERIMENTS, overwrite=False)
{key: {group: str(failure['error'])[:200] for group, failure in groups.items()} for key, groups in ice_failures.items()}

In [ ]:
ice_tree = loading.open_lesfmip_monthly(SEA_ICE, models=ice_models, experiments=config.MAIN_EXPERIMENTS,
                                        groups=[config.GROUP])


def ice_check(ds):
    #(t): From one member's first year: the largest concentration, and the continent the edge is measured from
    siconc = ds[SEA_ICE].isel(member=0, time=slice(0, 12)).load()
    area = sea_ice.cell_area(siconc.lat, siconc.lon)
    continent = float(sea_ice.continent_area(sea_ice.land_mask(siconc), area))
    return {'largest concentration (%)': float(siconc.max()), 'continent (million km²)': continent / 1e6,
            'edge with no ice (°)': float(sea_ice.equivalent_latitude(continent))}


#(t): Checks: the largest concentration should be near 100 (a model storing fractions would show 1), and the
#(t): continent about 14 million km², an edge near 71°S with no ice. A continent near 0 means land is stored as
#(t): 0% rather than missing, and the edges would be far too far south.
pd.DataFrame({node.parent.name: ice_check(node.to_dataset()) for node in ice_tree.match('*/hist-nat').leaves}).T.round(2)

**The method on one field.** One hist-nat member's first winter (JJA), step by step: the concentration and its 15% contour; the cells that count (ice and the continent), with the circle that encloses the same area; and the cap area $A(\varphi)$, read backwards from their total.

In [ ]:
ice_member_da = pp.to_analysis_units(ice_tree[config.PLOT_SEL['model']]['hist-nat'].to_dataset(), SEA_ICE)[SEA_ICE].isel(member=0)
first_winter_da = ice_member_da.isel(time=slice(0, 12)).where(ice_member_da.time.dt.month.isin([6, 7, 8]), drop=True).mean('time')
ice_example_ds = sea_ice.edge_example(first_winter_da.load(), land=sea_ice.land_mask(ice_member_da).load())
fig, axes = sip.edge_schematic(ice_example_ds, title=f'How the ice edge becomes a latitude: {config.PLOT_SEL["model"]}, '
                                                  'hist-nat, one member, JJA')

**All of it.** Each member's seasonal extent (only complete seasons in whole years, cut at `config.LAST_MONTH` like every other variable, so "the final years" are the same years), and the ensemble-mean concentration over each experiment's final `config.WINDOW` years.

In [ ]:
%%time
ice_extents_tree = sea_ice.seasonal_extent(ice_tree, SEA_ICE)
storage.save(ice_extents_tree, paths.result('extent', SEA_ICE, 'full'))
ice_climatology_da = sea_ice.concentration_climatology(pp.seasonal_tree(ice_tree, SEA_ICE), years=config.WINDOW,
                                                    variable=SEA_ICE)
storage.save(ice_climatology_da.rename(SEA_ICE), paths.result('climatology', SEA_ICE, 'full'))

In [ ]:
#(t): Check: the extent through time, and each model's edge (equivalent latitude) over the final WINDOW years
ice_extents_tree = storage.open_tree(paths.result('extent', SEA_ICE, 'full'), load=True)
fig, axes = sip.extent_timeseries(sea_ice.extent_summary(ice_extents_tree))
sea_ice.edge_latitude(ice_extents_tree).to_series().unstack('season').round(1)

## Moving the outputs to the group workspace

Once a store is written and checked, move it from scratch to the group workspace, keeping the same layout, e.g.

```bash
mkdir -p /gws/ssde/j25b/leader_epesc/aborow/ExtAnt/seasonal
mv /work/scratch-pw4/aborow/ExtAnt/seasonal/lesfmip_tas.zarr /gws/ssde/j25b/leader_epesc/aborow/ExtAnt/seasonal/
```

or everything at once with `rsync -a --remove-source-files /work/scratch-pw4/aborow/ExtAnt/ /gws/ssde/j25b/leader_epesc/aborow/ExtAnt/`. Nothing needs changing in the notebooks afterwards: they read from the group workspace first. If a store is rewritten later, the new copy lands on scratch again and a warning says the old one in the workspace is still read first until it is replaced.